In [162]:
import pandas as pd
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from typing import Literal
from pydantic import BaseModel, Field

In [145]:
with open('impact_evaluation_results_justif.pkl', 'rb') as f:
    results = pickle.load(f)

In [146]:
ok = 0
nok = 0
for idx, policy, impact, question, evaluation in results:
    if isinstance(evaluation, dict):
        ok += 1
        continue
    if evaluation.startswith('error'):
        nok += 1

ok, nok, len(results), ok + nok

(8735, 5, 8740, 8740)

In [147]:
results[42]

(9,
 'Housing Cooperatives and Community-Led Housing Models',
 'justice_considerations',
 'What is the impact of Housing Cooperatives and Community-Led Housing Models on justice considerations? Consider the following types of justice: distributional, procedural, corrective, recognitional, transitional.',
 {'evidences': [{'evidence': "Housing cooperatives and community-led housing models promote distributional justice by collectivizing risks and care work, providing long-term security through grant-of-use models, and ensuring access to housing and shared facilities for members. These models help reduce socio-economic disparities by aligning housing access with members' financial capacities and political values, particularly benefiting lower-income groups.",
    'impact': 'positive',
    'openalex_ids': ['W4310397208', 'W4295664119']},
   {'evidence': 'These models enhance procedural justice by enabling participatory governance, consensus-based decision-making, and inclusive processes wh

In [148]:
d = pd.DataFrame(results, columns=['idx', 'policy', 'impact', 'question', 'evaluation'])
d

,idx,policy,impact,question,evaluation
0,3,Operational Energy Optimization and HVAC Contr...,planetary_boundaries,What is the impact of Operational Energy Optim...,{'evidences': [{'evidence': 'The documents pro...
1,3,Operational Energy Optimization and HVAC Contr...,natural_resources,What is the impact of Operational Energy Optim...,{'evidences': [{'evidence': 'Operational Energ...
2,1,Energy Efficiency Standards and Codes for New ...,wellbeing,What is the impact of Energy Efficiency Standa...,{'evidences': [{'evidence': 'Energy Efficiency...
3,1,Energy Efficiency Standards and Codes for New ...,natural_resources,What is the impact of Energy Efficiency Standa...,{'evidences': [{'evidence': 'Energy requiremen...
4,0,Building Energy Performance Certification and ...,natural_resources,What is the impact of Building Energy Performa...,{'evidences': [{'evidence': 'Document 7 discus...
...,...,...,...,...,...
8735,40,Soil and Foundation Stabilization Using Chemic...,wellbeing,What is the impact of Soil and Foundation Stab...,error:http:500:Internal Server Error
8736,134,Floating Offshore Wind Innovation Incentives,wellbeing,What is the impact of Floating Offshore Wind I...,{'evidences': [{'evidence': 'The development o...
8737,1840,Multi-stakeholder collaboration and infrastruc...,wellbeing,What is the impact of Multi-stakeholder collab...,error:http:500:Internal Server Error
8738,2031,"Coastal Defense Infrastructure (Seawalls, Dike...",wellbeing,What is the impact of Coastal Defense Infrastr...,error:http:500:Internal Server Error


In [149]:
pdf = pd.read_parquet('data/sample_10k_policy_clusters_llm_2026-02-12.parquet')
pdf

,domain,subdomain,cluster
0,BUILDING,Building Energy Efficiency and Decarbonization,Building Energy Performance Certification and ...
1,BUILDING,Building Energy Efficiency and Decarbonization,Energy Efficiency Standards and Codes for New ...
2,BUILDING,Building Energy Efficiency and Decarbonization,Thermal and Envelope Retrofit Programs for Exi...
3,BUILDING,Building Energy Efficiency and Decarbonization,Operational Energy Optimization and HVAC Contr...
4,BUILDING,Building Energy Efficiency and Decarbonization,Embodied Energy and Life Cycle Assessment (LCA...
...,...,...,...
2180,URBAN,Built Environment Safety and Resilience,"Building safety and resilience (e.g., earthqua..."
2181,URBAN,Built Environment Safety and Resilience,"Material and design innovation (e.g., reflecti..."
2182,URBAN,Rural Development and Livelihood Support,"Rural infrastructure development (e.g., roads,..."
2183,URBAN,Rural Development and Livelihood Support,Support for aging farmers and flexible employm...


In [150]:
pivoted = d.set_index(['idx', 'impact'])['evaluation'].unstack()
pivoted

impact,justice_considerations,natural_resources,planetary_boundaries,wellbeing
idx,,,,
0,{'evidences': [{'evidence': 'Building Energy P...,{'evidences': [{'evidence': 'Document 7 discus...,{'evidences': [{'evidence': 'Building Energy P...,{'evidences': [{'evidence': 'Building Energy P...
1,{'evidences': [{'evidence': 'The paper discuss...,{'evidences': [{'evidence': 'Energy requiremen...,{'evidences': [{'evidence': 'Energy efficiency...,{'evidences': [{'evidence': 'Energy Efficiency...
2,{'evidences': [{'evidence': 'The study identif...,{'evidences': [{'evidence': 'Thermal and envel...,{'evidences': [{'evidence': 'Thermal and envel...,{'evidences': [{'evidence': 'Thermal retrofitt...
3,{'evidences': [{'evidence': 'The paper analyze...,{'evidences': [{'evidence': 'Operational Energ...,{'evidences': [{'evidence': 'The documents pro...,{'evidences': [{'evidence': 'Operational Energ...
4,{'evidences': [{'evidence': 'Life Cycle Assess...,{'evidences': [{'evidence': 'The use of wood s...,{'evidences': [{'evidence': 'Life Cycle Assess...,{'evidences': [{'evidence': 'The documents dis...
...,...,...,...,...
2180,{'evidences': [{'evidence': 'Building codes an...,"{'evidences': [], 'overall_impact': 'neutral'}",{'evidences': [{'evidence': 'Building safety a...,{'evidences': [{'evidence': 'Earthquake-resist...
2181,{'evidences': [{'evidence': 'Procedural justic...,{'evidences': [{'evidence': 'Technological inn...,{'evidences': [{'evidence': 'The provided docu...,{'evidences': [{'evidence': 'Natural materials...
2182,{'evidences': [{'evidence': 'Rural infrastruct...,{'evidences': [{'evidence': 'The degradative i...,{'evidences': [{'evidence': 'The development o...,{'evidences': [{'evidence': 'Rural infrastruct...


In [151]:
jdf = pdf.join(pivoted).dropna()
jdf

,domain,subdomain,cluster,justice_considerations,natural_resources,planetary_boundaries,wellbeing
0,BUILDING,Building Energy Efficiency and Decarbonization,Building Energy Performance Certification and ...,{'evidences': [{'evidence': 'Building Energy P...,{'evidences': [{'evidence': 'Document 7 discus...,{'evidences': [{'evidence': 'Building Energy P...,{'evidences': [{'evidence': 'Building Energy P...
1,BUILDING,Building Energy Efficiency and Decarbonization,Energy Efficiency Standards and Codes for New ...,{'evidences': [{'evidence': 'The paper discuss...,{'evidences': [{'evidence': 'Energy requiremen...,{'evidences': [{'evidence': 'Energy efficiency...,{'evidences': [{'evidence': 'Energy Efficiency...
2,BUILDING,Building Energy Efficiency and Decarbonization,Thermal and Envelope Retrofit Programs for Exi...,{'evidences': [{'evidence': 'The study identif...,{'evidences': [{'evidence': 'Thermal and envel...,{'evidences': [{'evidence': 'Thermal and envel...,{'evidences': [{'evidence': 'Thermal retrofitt...
3,BUILDING,Building Energy Efficiency and Decarbonization,Operational Energy Optimization and HVAC Contr...,{'evidences': [{'evidence': 'The paper analyze...,{'evidences': [{'evidence': 'Operational Energ...,{'evidences': [{'evidence': 'The documents pro...,{'evidences': [{'evidence': 'Operational Energ...
4,BUILDING,Building Energy Efficiency and Decarbonization,Embodied Energy and Life Cycle Assessment (LCA...,{'evidences': [{'evidence': 'Life Cycle Assess...,{'evidences': [{'evidence': 'The use of wood s...,{'evidences': [{'evidence': 'Life Cycle Assess...,{'evidences': [{'evidence': 'The documents dis...
...,...,...,...,...,...,...,...
2180,URBAN,Built Environment Safety and Resilience,"Building safety and resilience (e.g., earthqua...",{'evidences': [{'evidence': 'Building codes an...,"{'evidences': [], 'overall_impact': 'neutral'}",{'evidences': [{'evidence': 'Building safety a...,{'evidences': [{'evidence': 'Earthquake-resist...
2181,URBAN,Built Environment Safety and Resilience,"Material and design innovation (e.g., reflecti...",{'evidences': [{'evidence': 'Procedural justic...,{'evidences': [{'evidence': 'Technological inn...,{'evidences': [{'evidence': 'The provided docu...,{'evidences': [{'evidence': 'Natural materials...
2182,URBAN,Rural Development and Livelihood Support,"Rural infrastructure development (e.g., roads,...",{'evidences': [{'evidence': 'Rural infrastruct...,{'evidences': [{'evidence': 'The degradative i...,{'evidences': [{'evidence': 'The development o...,{'evidences': [{'evidence': 'Rural infrastruct...
2183,URBAN,Rural Development and Livelihood Support,Support for aging farmers and flexible employm...,{'evidences': [{'evidence': 'The provided docu...,{'evidences': [{'evidence': 'Support for aging...,{'evidences': [{'evidence': 'Support for aging...,{'evidences': [{'evidence': 'Older farmers are...


In [152]:
impacts = d['impact'].unique().tolist()
impacts

['planetary_boundaries',
 'natural_resources',
 'wellbeing',
 'justice_considerations']

In [153]:
def impact_summary(evidences_dict : dict, impact: str) -> str:
    s = f"""Impact on : {impact.replace('_', ' ')}
Overall evaluation : {evidences_dict['overall_impact']}
Evidences :
"""
    for ev in evidences_dict['evidences']:
        s += f" - {ev['impact']}: {ev['evidence']} ({', '.join(ev['openalex_ids'])}) \n"
    
    return s.strip()

def summarize_impacts(row):
    s = f"""Domain : {row['domain']}
Subdomain : {row['subdomain']}
Policy : {row['cluster']}\n\n"""
    for impact in impacts:
        evidences = row[impact]
        if isinstance(evidences, dict):
            s += impact_summary(evidences, impact) + "\n\n"
        else:
            print(f"No impact for {impact} and policy {row['cluster']}")
            return None  # prefer to return None if any of the impacts is missing, to avoid having partial summaries
    return s.strip()


row = jdf.iloc[0]
print(summarize_impacts(row))

Domain : BUILDING
Subdomain : Building Energy Efficiency and Decarbonization
Policy : Building Energy Performance Certification and Regulation

Impact on : planetary boundaries
Overall evaluation : positive
Evidences :
 - positive: Building Energy Performance Certification and Regulation contributes to reducing energy consumption and greenhouse gas emissions from buildings, which directly addresses climate change by lowering CO2 emissions. This is supported by policies like the EU directive on energy performance of buildings and national strategies aimed at reducing energy demands, which help keep atmospheric CO2 concentrations within planetary boundaries. (W3098232646) 
 - positive: By improving energy efficiency in buildings, such regulations indirectly reduce the need for extensive land use for energy production, thus helping to maintain land system change within the 15% cropland threshold of ice-free land. This is inferred from the general impact of energy efficiency on reducing re

In [154]:
jdf['impact_summary'] = jdf.apply(summarize_impacts, axis=1)

No impact for wellbeing and policy Soil and Foundation Stabilization Using Chemical and Mechanical Methods
No impact for wellbeing and policy Floating Offshore Wind Innovation Incentives
No impact for planetary_boundaries and policy Green growth and low-carbon transition
No impact for wellbeing and policy Multi-stakeholder collaboration and infrastructure investment for circularity
No impact for wellbeing and policy Coastal Defense Infrastructure (Seawalls, Dikes, Living Shorelines)


In [160]:
print(jdf.loc[0, 'impact_summary'])

Domain : BUILDING
Subdomain : Building Energy Efficiency and Decarbonization
Policy : Building Energy Performance Certification and Regulation

Impact on : planetary boundaries
Overall evaluation : positive
Evidences :
 - positive: Building Energy Performance Certification and Regulation contributes to reducing energy consumption and greenhouse gas emissions from buildings, which directly addresses climate change by lowering CO2 emissions. This is supported by policies like the EU directive on energy performance of buildings and national strategies aimed at reducing energy demands, which help keep atmospheric CO2 concentrations within planetary boundaries. (W3098232646) 
 - positive: By improving energy efficiency in buildings, such regulations indirectly reduce the need for extensive land use for energy production, thus helping to maintain land system change within the 15% cropland threshold of ice-free land. This is inferred from the general impact of energy efficiency on reducing re

In [198]:
sufficiency_eval_prompt = """
Sufficiency is a set of policy measures and daily practices which avoid the demand for energy, materials, land, water, and other natural resources,
while delivering wellbeing for all within planetary boundaries.
Importantly, sufficiency isn't efficiency, which is doing more or the same with less.
Sufficiency is about *avoiding* demand. Also sufficiency entails both a physical ceiling and a social floor.

The following impact dimensions should be considered when evaluating whether a policy is a sufficiency policy or not:
- natural resources: freshwater, marine resources, wetlands, metals and ores, non-metallic minerals, fossil fuels, agricultural land, forests, urban land, biomass.
- wellbeing: housing, jobs, education, civic engagement, life satisfaction, work-life balance, income, community, environment, health, safety.
- justice: distributional, procedural, corrective, recognitional, transitional.
- planetary boundaries: land system change, climate change, biosphere integrity, biogeochemical flows, ocean acidification, freshwater use, atmospheric aerosol loading, ozone depletion, introduction of novel entities.

You will find below a policy along with retrieved evidences of its impacts on the above dimensions.

Classify the policy into one of the below three categories:
- Sufficiency (S): Primarily and directly contributes to human well-being while significantly reducing resource demand and staying within planetary boundaries.
- Potential Sufficiency (PS): Has the potential to achieve sufficiency, but requires explicit, specific, and significant transformation (e.g., equity corrections, policy integration) to overcome limitations and align with all boundaries.
- Not Sufficiency (NS): The policy's effect on human needs is indirect, often resulting in breaches of planetary limits or social foundations. The policy might turn up to be a violator of basic needs.  Or intrinsically undermines one or both boundaries (social foundations or planetary limits).

Don't fall into the trap of classifying everything as PS. Don't expect absolutely everything to be perfect for S, or to be bad for NS.
"""


def build_messages(impact_summary: str) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": sufficiency_eval_prompt},
        {"role": "user", "content": f"Policy impact summary : \n\n {impact_summary}"}
    ]


class SufficiencyClassificationResponse(BaseModel):
    reasoning: str = Field(description="Short reasoning justifying the classification based on the provided evidence. Mention openalex ids to support your arguments when relevant.")
    classification: Literal['S', 'PS', 'NS']

In [189]:
from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()

True

In [190]:
client = OpenAI(
    base_url=os.getenv("GENERATION_API_URL"),
    api_key=os.getenv("SCW_SECRET_KEY"),
)

In [191]:
model = "qwen3-235b-a22b-instruct-2507"

In [193]:
def classify_sufficiency(evidences_summary: str):
    try:
        kwargs = {
            "model": model,
            "messages": build_messages(evidences_summary),
            "max_tokens": 1024,
            "temperature": 0.05,
            "top_p": 0.1,
            "timeout": 60,
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                    "name": SufficiencyClassificationResponse.__name__,
                    "schema": SufficiencyClassificationResponse.model_json_schema(),
                },
            },
        }

        response = client.chat.completions.create(**kwargs)
        return SufficiencyClassificationResponse.model_validate_json(
            response.choices[0].message.content
        )
    except Exception as e:
        print(f"Error generating response: {e}")

In [ ]:
r = classify_sufficiency(jdf.iloc[651]['impact_summary'])

In [205]:
jdf.iloc[651].cluster

'Regulatory sandboxes and risk monitoring'

In [200]:
r

SufficiencyClassificationResponse(reasoning='The policy of regulatory sandboxes and risk monitoring shows a positive impact on natural resources through enabling better regulation and monitoring, which can prevent overexploitation and support sustainable resource management. However, there is insufficient evidence regarding its direct impact on wellbeing and justice considerations, and no clear link to planetary boundaries. While the policy has the potential to contribute to sufficiency by improving environmental governance and preventing resource overuse, it currently lacks explicit integration with social floor guarantees and comprehensive planetary boundary alignment. Therefore, it qualifies as Potential Sufficiency (PS), requiring further development and equity-focused implementation to fully realize sufficiency goals.', classification='PS')

In [212]:
# parallelize the classification of sufficiency for all policies
results = []
with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {executor.submit(classify_sufficiency, row['impact_summary']): idx for idx, row in jdf.iterrows()}
    for future in tqdm(as_completed(futures), total=len(futures)):
        idx = futures[future]
        try:
            result = future.result()
            if isinstance(result, SufficiencyClassificationResponse):
                results.append((idx, result.reasoning, result.classification))
            else:
                print(f"Invalid response for index {idx}: {result}")
        except Exception as e:
            print(f"Error processing index {idx}: {e}")

  0%|          | 0/2185 [00:00<?, ?it/s]

In [220]:
final = pd.DataFrame(results, columns=['idx', 'sufficiency_classification_reasoning', 'sufficiency_classification'])
final = final.set_index('idx')
final

,sufficiency_classification_reasoning,sufficiency_classification
idx,,
25,The policy on Indoor Environmental Quality (IE...,S
19,The policy on Structural Health Monitoring and...,NS
5,The policy of Affordable and Social Housing Pr...,S
23,The policy of prefabrication and industrialize...,PS
8,The policy on post-disaster and reconstruction...,PS
...,...,...
2175,The policy of urban forestry standards and bio...,PS
2180,The policy of building safety and resilience (...,PS
2178,The policy of fiscal incentives and insurance-...,NS


In [223]:
real_final = jdf.join(final)

In [225]:
real_final

,domain,subdomain,cluster,justice_considerations,natural_resources,planetary_boundaries,wellbeing,impact_summary,sufficiency_classification_reasoning,sufficiency_classification
0,BUILDING,Building Energy Efficiency and Decarbonization,Building Energy Performance Certification and ...,{'evidences': [{'evidence': 'Building Energy P...,{'evidences': [{'evidence': 'Document 7 discus...,{'evidences': [{'evidence': 'Building Energy P...,{'evidences': [{'evidence': 'Building Energy P...,Domain : BUILDING\nSubdomain : Building Energy...,The policy of Building Energy Performance Cert...,PS
1,BUILDING,Building Energy Efficiency and Decarbonization,Energy Efficiency Standards and Codes for New ...,{'evidences': [{'evidence': 'The paper discuss...,{'evidences': [{'evidence': 'Energy requiremen...,{'evidences': [{'evidence': 'Energy efficiency...,{'evidences': [{'evidence': 'Energy Efficiency...,Domain : BUILDING\nSubdomain : Building Energy...,The policy of Energy Efficiency Standards and ...,PS
2,BUILDING,Building Energy Efficiency and Decarbonization,Thermal and Envelope Retrofit Programs for Exi...,{'evidences': [{'evidence': 'The study identif...,{'evidences': [{'evidence': 'Thermal and envel...,{'evidences': [{'evidence': 'Thermal and envel...,{'evidences': [{'evidence': 'Thermal retrofitt...,Domain : BUILDING\nSubdomain : Building Energy...,The policy of thermal and envelope retrofit pr...,S
3,BUILDING,Building Energy Efficiency and Decarbonization,Operational Energy Optimization and HVAC Contr...,{'evidences': [{'evidence': 'The paper analyze...,{'evidences': [{'evidence': 'Operational Energ...,{'evidences': [{'evidence': 'The documents pro...,{'evidences': [{'evidence': 'Operational Energ...,Domain : BUILDING\nSubdomain : Building Energy...,The policy of Operational Energy Optimization ...,PS
4,BUILDING,Building Energy Efficiency and Decarbonization,Embodied Energy and Life Cycle Assessment (LCA...,{'evidences': [{'evidence': 'Life Cycle Assess...,{'evidences': [{'evidence': 'The use of wood s...,{'evidences': [{'evidence': 'Life Cycle Assess...,{'evidences': [{'evidence': 'The documents dis...,Domain : BUILDING\nSubdomain : Building Energy...,The policy of incorporating Embodied Energy an...,PS
...,...,...,...,...,...,...,...,...,...,...
2180,URBAN,Built Environment Safety and Resilience,"Building safety and resilience (e.g., earthqua...",{'evidences': [{'evidence': 'Building codes an...,"{'evidences': [], 'overall_impact': 'neutral'}",{'evidences': [{'evidence': 'Building safety a...,{'evidences': [{'evidence': 'Earthquake-resist...,Domain : URBAN\nSubdomain : Built Environment ...,The policy of building safety and resilience (...,PS
2181,URBAN,Built Environment Safety and Resilience,"Material and design innovation (e.g., reflecti...",{'evidences': [{'evidence': 'Procedural justic...,{'evidences': [{'evidence': 'Technological inn...,{'evidences': [{'evidence': 'The provided docu...,{'evidences': [{'evidence': 'Natural materials...,Domain : URBAN\nSubdomain : Built Environment ...,The policy of material and design innovation (...,PS
2182,URBAN,Rural Development and Livelihood Support,"Rural infrastructure development (e.g., roads,...",{'evidences': [{'evidence': 'Rural infrastruct...,{'evidences': [{'evidence': 'The degradative i...,{'evidences': [{'evidence': 'The development o...,{'evidences': [{'evidence': 'Rural infrastruct...,Domain : URBAN\nSubdomain : Rural Development ...,The policy of rural infrastructure development...,NS
2183,URBAN,Rural Development and Livelihood Support,Support for aging farmers and flexible employm...,{'evidences': [{'evidence': 'The provided docu...,{'evidences': [{'evidence': 'Support for aging...,{'evidences': [{'evidence': 'Support for aging...,{'evidences': [{'evidence': 'Older farmers are...,Domain : URBAN\nSubdomain : Rural Development ...,The policy of supporting aging farmers and fle...,S


In [228]:
real_final.dropna().to_parquet('sample10k_sufficiency_classification_results_2026-02-18.parquet')

In [233]:
real_final.drop(columns=[i for i in impacts]).to_csv('sample10k_sufficiency_classification_results_2026-02-18.csv')

In [ ]:
real_final.to_csv('sample10k_sufficiency_classification_results_2026-02-18.csv')

In [230]:
real_final.sufficiency_classification.value_counts()

sufficiency_classification
PS    821
NS    702
S     662
Name: count, dtype: int64